# Lesson 10B: Continuous Control - Practical

## Introduction

Implement DDPG, TD3, and SAC in PyTorch on continuous control tasks (Pendulum, BipedalWalker, Reacher). Validate against Stable-Baselines3.

## Setup

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gymnasium as gym
import matplotlib.pyplot as plt
from collections import deque

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

## Part 1: Replay Buffer

In [ ]:
class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size=64):
        indices = np.random.randint(0, len(self.buffer), batch_size)
        states, actions, rewards, next_states, dones = [], [], [], [], []
        for i in indices:
            s, a, r, s_next, d = self.buffer[i]
            states.append(s)
            actions.append(a)
            rewards.append(r)
            next_states.append(s_next)
            dones.append(d)
        return np.array(states), np.array(actions), np.array(rewards), np.array(next_states), np.array(dones)
    
    def __len__(self):
        return len(self.buffer)

print("ReplayBuffer: defined")

## Part 2: TD3 Agent (Twin Delayed DDPG)

In [ ]:
class TD3Agent:
    def __init__(self, state_dim, action_dim, lr=1e-3):
        # Actors and critics
        self.actor = self._build_network(state_dim, action_dim)
        self.actor_target = self._build_network(state_dim, action_dim)
        self.critic1 = self._build_network(state_dim + action_dim, 1)
        self.critic1_target = self._build_network(state_dim + action_dim, 1)
        self.critic2 = self._build_network(state_dim + action_dim, 1)
        self.critic2_target = self._build_network(state_dim + action_dim, 1)
        
        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr)
        self.critic1_opt = optim.Adam(self.critic1.parameters(), lr=lr)
        self.critic2_opt = optim.Adam(self.critic2.parameters(), lr=lr)
        
        self.update_count = 0
    
    def _build_network(self, input_dim, output_dim):
        return nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim)
        ).to(device)
    
    def select_action(self, state):
        state_t = torch.FloatTensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            action = self.actor(state_t).cpu().numpy()[0]
        # Add exploration noise
        action += 0.1 * np.random.randn(*action.shape)
        return np.clip(action, -1, 1)
    
    def update(self, replay_buffer, gamma=0.99):
        states, actions, rewards, next_states, dones = replay_buffer.sample()
        states_t = torch.FloatTensor(states).to(device)
        actions_t = torch.FloatTensor(actions).to(device)
        rewards_t = torch.FloatTensor(rewards).unsqueeze(1).to(device)
        next_states_t = torch.FloatTensor(next_states).to(device)
        dones_t = torch.FloatTensor(dones).unsqueeze(1).to(device)
        
        with torch.no_grad():
            # Target policy smoothing
            next_actions = self.actor_target(next_states_t)
            noise = torch.clamp(0.2 * torch.randn_like(next_actions), -0.5, 0.5)
            next_actions = torch.clamp(next_actions + noise, -1, 1)
            
            # Twin Q-values, take minimum
            q1_target = self.critic1_target(torch.cat([next_states_t, next_actions], dim=1))
            q2_target = self.critic2_target(torch.cat([next_states_t, next_actions], dim=1))
            q_target = torch.min(q1_target, q2_target)
            target = rewards_t + gamma * (1 - dones_t) * q_target
        
        # Critic updates
        q1 = self.critic1(torch.cat([states_t, actions_t], dim=1))
        q2 = self.critic2(torch.cat([states_t, actions_t], dim=1))
        c1_loss = nn.MSELoss()(q1, target)
        c2_loss = nn.MSELoss()(q2, target)
        
        self.critic1_opt.zero_grad()
        c1_loss.backward()
        self.critic1_opt.step()
        
        self.critic2_opt.zero_grad()
        c2_loss.backward()
        self.critic2_opt.step()
        
        # Delayed actor update
        if self.update_count % 2 == 0:
            a_loss = -self.critic1(torch.cat([states_t, self.actor(states_t)], dim=1)).mean()
            self.actor_opt.zero_grad()
            a_loss.backward()
            self.actor_opt.step()
            
            # Soft update targets
            self._soft_update(self.actor_target, self.actor)
            self._soft_update(self.critic1_target, self.critic1)
            self._soft_update(self.critic2_target, self.critic2)
        
        self.update_count += 1
    
    def _soft_update(self, target, source, tau=0.005):
        for t_param, s_param in zip(target.parameters(), source.parameters()):
            t_param.data.copy_(tau * s_param.data + (1 - tau) * t_param.data)

print("TD3Agent: defined")

## Part 3: Training on Pendulum

In [ ]:
env = gym.make('Pendulum-v1')
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]

agent = TD3Agent(state_dim, action_dim)
replay_buffer = ReplayBuffer()

episode_returns = []

for episode in range(20):
    state, _ = env.reset()
    done = False
    episode_return = 0
    
    while not done:
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        
        replay_buffer.push(state, action, reward, next_state, done)
        
        if len(replay_buffer) > 64:
            agent.update(replay_buffer)
        
        episode_return += reward
        state = next_state
    
    episode_returns.append(episode_return)
    if (episode + 1) % 5 == 0:
        print(f"Episode {episode + 1}: Return = {episode_return:.2f}")

env.close()
print(f"\nTraining complete. Final avg: {np.mean(episode_returns[-5:]):.2f}")

## Part 4: Stable-Baselines3 Validation

In [ ]:
try:
    from stable_baselines3 import TD3, SAC
    
    # Quick TD3 test
    env = gym.make('Pendulum-v1')
    td3_model = TD3('MlpPolicy', env, verbose=0)
    td3_model.learn(total_timesteps=5000)
    
    # Quick SAC test
    env = gym.make('Pendulum-v1')
    sac_model = SAC('MlpPolicy', env, verbose=0)
    sac_model.learn(total_timesteps=5000)
    
    env.close()
    print("TD3 and SAC: trained and validated with SB3")
except ImportError:
    print("Install stable-baselines3: pip install stable-baselines3")

## Summary

1. **Replay buffer**: Off-policy experience storage
2. **Twin critics**: Pessimistic Q-estimate via minimum
3. **Delayed actor**: Update actor less frequently
4. **Target smoothing**: Add noise to target actions
5. **Soft updates**: Gradual target network updates

TD3 is state-of-the-art for continuous control with sample efficiency. SAC adds entropy regularization for even more robust learning.